# 01 — Data audit

Day 8 checks against `data/myriad.duckdb`.

**Acceptance (prd.md):**
- ≥15 seasons of results
- ≥10 seasons with xG
- zero unresolved team names
- zero duplicate `match_id`s

Re-run any time with `\.\tasks.ps1 audit` or the cells below.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path("..").resolve()
DB = ROOT / "data" / "myriad.duckdb"
assert DB.exists(), "Run .\\tasks.ps1 load first"
con = duckdb.connect(str(DB), read_only=True)
print("connected:", DB)

## Matches per season

In [ ]:
by_season = con.execute("""
SELECT season, COUNT(*) AS matches,
       SUM(CASE WHEN home_xg IS NOT NULL THEN 1 ELSE 0 END) AS with_xg,
       SUM(CASE WHEN odds_open_h IS NOT NULL THEN 1 ELSE 0 END) AS with_open_odds,
       SUM(CASE WHEN odds_close_h IS NOT NULL THEN 1 ELSE 0 END) AS with_close_odds
FROM matches
GROUP BY 1 ORDER BY 1
""").fetchdf()
display(by_season)
print(f"seasons={len(by_season)}  xg_seasons={(by_season['with_xg']>0).sum()}")

## Duplicates, null team ids, fixtures

In [ ]:
null_teams = con.execute("""
SELECT COUNT(*) FROM matches
WHERE home_team_id IS NULL OR away_team_id IS NULL
   OR home_team_id = '' OR away_team_id = ''
""").fetchone()[0]

dups = con.execute("""
SELECT match_id, COUNT(*) n FROM matches
GROUP BY 1 HAVING COUNT(*) > 1
""").fetchdf()

fixtures = con.execute("""
SELECT season, COUNT(*) n, MIN(kickoff_utc) first_ko, MAX(kickoff_utc) last_ko
FROM fixtures GROUP BY 1
""").fetchdf()

print("null team_ids:", null_teams)
print("duplicate match_ids:", len(dups))
display(fixtures)

ok = (
    len(by_season) >= 15
    and (by_season["with_xg"] > 0).sum() >= 10
    and null_teams == 0
    and len(dups) == 0
)
print("ACCEPTANCE:", "PASSED" if ok else "FAILED")